# 🛒 Purchase Propensity + RFM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/purchase-propensity/training/notebook.ipynb)

Two classic e-commerce analytics tasks in one project:

- **Purchase propensity**: will a website visitor place an order? 455,401 one-day sessions from the
  [Kaggle propensity data](https://www.kaggle.com/datasets/benpowis/customer-propensity-to-purchase-data), each
  described by 23 binary flags (added to basket, saw checkout, signed in, device, ...). Only 4.19 % of sessions order.
  Two experiments are compared: a **LogisticRegression** with balanced class weights (24 parameters) and a
  **PyTorch MLP** 23 → 64 → 32 → 1 (3,649 parameters). Each gets a decision threshold tuned for max F1 on the
  validation split; the default model is the one with the best **validation PR-AUC**, and the test split is scored once.
- **RFM segmentation**: the [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail) invoices are
  turned into Recency / Frequency / Monetary quintile scores per customer and mapped onto the standard segment grid
  (Champions, Loyal Customers, At Risk, Hibernating, ...).

Both models, the scaler and the segment table are exported to `../model/` and served by the
[Space](https://huggingface.co/spaces/shalev396/purchase-propensity). All logic lives in `src/` and `../model/model.py`;
this notebook calls it step by step.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "purchase-propensity"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# 1d. Project code: src/ (training helpers) and ../model/model.py (architecture + Predictor, same file the Space runs)
import time

import pandas as pd
from IPython.display import display

from src import data_setup, engine, export, model_builder, utils
from src.config import Config
from src.model_builder import LOGREG, MLP

import model as M  # ../model/model.py, importable once `src` is imported

print(utils.lib_versions("torch", "sklearn", "numpy", "pandas", "huggingface_hub"))

## 2. Config
Every hyperparameter lives in `src/config.py`. `SMOKE_TEST=1` runs the same pipeline on a 25k-session stratified
subset with a tiny MLP (16 hidden units, 3 epochs) and exports to `outputs/smoke/model/` instead of `../model/`.

In [ ]:
# "0" = full run (~3 min on a desktop CPU), "1" = smoke run. setdefault keeps a value exported by the caller
# (e.g. `SMOKE_TEST=1 jupyter nbconvert --execute ...`); restart the kernel after changing it.
os.environ.setdefault("SMOKE_TEST", "0")
PUSH_TO_HUB = False  # upload the exported model repo to the Hub at the end (section 8)

cfg = Config()
utils.set_seeds(cfg.seed, "torch")
cfg

## 3. Data
**Sessions** (Data A): downloaded once with kagglehub into `data/`. Stratified split 68 / 12 / 20 (train / val / test);
a `StandardScaler` is fit on the train split only and used by both models.

In [ ]:
sessions = data_setup.load_sessions(cfg)
splits = data_setup.split(sessions, cfg)
print(f"{len(sessions):,} sessions, {sessions[M.TARGET].sum():,} orders ({sessions[M.TARGET].mean():.2%}), "
      f"{sessions[M.FEATURES].drop_duplicates().shape[0]:,} distinct flag combinations")
display(data_setup.split_summary(splits))
sessions.head()

Order rate with each flag ON vs OFF (train split). A few flags, such as `saw_checkout` and
`checked_delivery_detail`, separate buyers almost perfectly: they happen late in the purchase funnel.

In [ ]:
rates = data_setup.order_rate_by_flag(splits.train)
ax = rates[["order_rate_on", "order_rate_off"]].plot.barh(figsize=(7, 7), color=export.SERIES[:2],
                                                           title="Order rate when the flag is ON vs OFF")
ax.invert_yaxis()
rates.round(4)

**Retail invoices** (Data B): UCI Online Retail, cleaned (no missing `CustomerID`, no cancellations, positive
quantity and price) and cached as parquet. Recency / Frequency / Monetary per customer → rank-based quintile
scores → segment on the (R, F) grid.

In [ ]:
retail = data_setup.load_retail()
rfm = data_setup.compute_rfm(retail)
rfm_summary = data_setup.summarize_segments(rfm)
rfm_stats = data_setup.rfm_artifact(retail, rfm, rfm_summary)
print(f"{len(retail):,} invoice lines, {rfm_stats['n_customers']:,} customers, snapshot {rfm_stats['snapshot']}")
display(rfm.head())
rfm_summary

## 4. Load model
The two experiments. The MLP class `PropensityMLP` lives in `../model/model.py` (a `PyTorchModelHubMixin`, so it
saves as `model.safetensors` + `config.json`).

In [ ]:
logreg = model_builder.build_logreg(cfg)
mlp = model_builder.build_mlp(cfg)
params = {LOGREG: model_builder.count_params(LOGREG, logreg),  # 23 coefficients + 1 intercept
          MLP: model_builder.count_params(MLP, mlp)}
print(logreg)
print(mlp)
pd.Series(params, name="parameters").to_frame()

## 5. Training
LogisticRegression on the standardized train split, then the MLP: `BCEWithLogitsLoss(pos_weight = negatives /
positives)`, AdamW, early stopping on validation ROC-AUC (the best epoch's weights are kept).

In [ ]:
start = time.perf_counter()
logreg, lr_seconds = engine.fit_logreg(logreg, splits)
print(f"logistic regression fit in {lr_seconds:.1f} s")

train_loader, val_loader = data_setup.make_loaders(splits, cfg.batch_size, device)
weight = engine.pos_weight(splits.xy("train")[1])
print(f"pos_weight = {weight:.2f}")
history = engine.train_mlp(mlp, train_loader, val_loader, splits.xy("val")[1], cfg, device, weight)
train_time_s = time.perf_counter() - start
print(f"total training time {train_time_s:.1f} s")

In [ ]:
export.plot_training_curves(history)

## 6. Evaluation
Each model's threshold is tuned for max F1 on **validation**; the default model is chosen by **validation PR-AUC**;
only then is the **test** split scored. ROC-AUC is the headline metric; PR-AUC and F1 matter more with 4 % positives.

In [ ]:
evals = {LOGREG: engine.evaluate(LOGREG, logreg, splits), MLP: engine.evaluate(MLP, mlp, splits, device)}
best = engine.select_best(evals)
print("default model (best validation PR-AUC):", best)
engine.comparison_table(evals, params).round(4)

In [ ]:
display(export.plot_roc_pr(evals, best, splits.test[M.TARGET].to_numpy()))
cm = engine.confusion(splits.test[M.TARGET].to_numpy(), evals[best].test_prob, evals[best].threshold)
print("test confusion:", cm)
export.plot_confusion(cm, best, evals[best].threshold)

What drives the scores: the logistic regression's coefficients (on standardized flags) and the MLP's
permutation importance (test ROC-AUC drop when one flag is shuffled).

In [ ]:
coefficients = engine.logreg_coefficients(logreg)
importance = engine.permutation_importance(MLP, mlp, splits, device, cfg.importance_repeats, cfg.seed)
export.plot_importance(coefficients, importance)

In [ ]:
export.plot_rfm_segments(rfm_summary)

## 7. Inference
Write the weights, reload the folder with `model.load(...)` and score the sessions in `../space/examples/`
(real test-split sessions): exactly what the Space and the Inference Endpoint run. `save_model` also checks that
the reloaded models reproduce the in-memory predictions.

In [ ]:
export.save_model(cfg.model_dir, scaler=splits.scaler, logreg=logreg, mlp=mlp, evals=evals, best=best,
                  params=params, rfm_stats=rfm_stats, splits=splits)
predictor = M.load(cfg.model_dir, device)

examples = pd.read_csv("../space/examples/visitors.csv")
active = [[f for f in M.FEATURES if row[f] == 1] for _, row in examples.iterrows()]
print(active[0], "->", predictor.predict(active[0]))
print("explain:", predictor.explain(active[0]))
batch = predictor.predict_batch(examples)
pd.DataFrame(batch["rows"]).assign(actual=examples[M.TARGET].map({0: "no order", 1: "order"}),
                                   active_flags=[", ".join(a) for a in active])

In [ ]:
# RFM: place a new customer on the segment grid (days since last order, number of orders, total spend in GBP)
predictor.rfm_segment(recency_days=12, frequency=6, monetary=2500)

## 8. Export
`metrics.json` (deployed model + both experiments), the card plots in `assets/` and the refreshed model card.
A full run also refreshes `../space/examples/visitors.csv`. Set `PUSH_TO_HUB = True` after a full run to upload
the folder; the token comes from `hf auth login` or an `HF_TOKEN` secret / environment variable. Smoke runs are
never uploaded.

In [ ]:
metrics = export.build_metrics(cfg, evals=evals, best=best, splits=splits, params=params, history=history,
                               rfm_stats=rfm_stats, device=str(device), train_time_s=train_time_s)
export.write_report(cfg.model_dir, metrics, evals=evals, splits=splits, history=history,
                    coefficients=coefficients, importance=importance, rfm_summary=rfm_summary)
if not cfg.smoke:
    print("examples ->", export.write_examples(splits))

for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():<32} {path.stat().st_size / 1e3:>9.1f} KB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))